# 🕵️‍♂️ Kiểm thử Model sau khi Fine-tune (Inference)

Sau khi mô hình chạy xong tiến trình huấn luyện ở file `02_finetuning_qlora.ipynb` và xuất ra thư mục **Adapter (LoRA weights)**, file Notebook này sẽ giúp bạn ghép nối nó vào Base Model gốc để xem kết quả chất lượng tóm tắt.

In [1]:
import sys
import os
import torch
from peft import PeftModel

# Khai báo hệ thống để import thư mục modules/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer
from modules.dataset_utils import format_prompt

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Bước 1: Nạp Base Model và Tokenizer
Chúng ta cần tải bản gốc chuẩn bị trước. (Tốc độ sẽ rất nhanh nếu `model_name` đã quét được từ trong cache/ổ cứng của bạn).

In [3]:
model_name = "Qwen/Qwen2.5-3B" # Thay bằng tên folder base model của bạn nếu cần tải nạp offline

base_model, tokenizer = load_model_and_tokenizer(
    model_name=model_name,
    use_4bit=True,
    torch_dtype="float16",
    device_map="auto"
)

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 253.07it/s]


### Bước 2: Nạp các tham số LoRA (Adapter) đã học được vào Base Model
Dùng `PeftModel` để khoác thêm "áo mới" lên model gốc.

In [4]:
lora_dir = os.path.join(PROJECT_ROOT, "models", "qwen2_5_meeting_lora")

try:
    model = PeftModel.from_pretrained(base_model, lora_dir)
    print("Đã dán Finetuned Adapter vào Base Model thành công!")
except Exception as e:
    print(f"Không tìm thấy thư mục LoRA tại: {lora_dir}. Quá trình train của bạn đã tạo ra file chưa?")
    raise e

Đã dán Finetuned Adapter vào Base Model thành công!


### Bước 3: Đưa câu hỏi và Test khả năng tóm tắt
Tôi sẽ viết ra một đoạn Text đóng vai trò là một cuộc họp giả lập.

In [9]:
# Bạn có thể sửa đoạn này thành nội dung một file text .txt thực tế của bạn
test_meeting_text = """
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Tốc độ lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| Soạn lại giáo trình đào tạo kỹ năng mềm cho nhân viên | Chị Lan (Phòng Nhân sự) | 25/05/2024 |
| Thiết kế lại sơ đồ quầy bar và báo giá thi công | Chị Mai (Phòng Vận hành) | 28/05/2024 |

[08:20] Ý tưởng hay đó Mai, vầy nè, em làm việc với bên thiết kế xem có phương án nào tối ưu không, rồi báo giá cho anh luôn nha.
[08:22] Dạ, để em triển khai cái này liền, chắc tầm cuối tuần là có bản vẽ sơ bộ cho anh coi đó.
[08:24] Còn một cái nữa, anh thấy khách họ cũng nhắc nhiều về cái vụ vệ sinh trong quán, nhất là cái nhà vệ sinh á, nhiều khi vô thấy dơ mà không có ai dọn dẹp thường xuyên.
[08:26] Ờ đúng rồi, cái này quan trọng nè. Em nghĩ mình nên có cái checklist dọn dẹp mỗi 30 phút một lần, rồi dán ở sau cửa luôn, bạn nào dọn xong thì ký tên vô đó để mình kiểm soát.
[08:28] Đúng rồi đó Lan, làm vậy đi cho nó chuyên nghiệp. À mà mình cũng nên có cái chương trình thưởng cho "Nhân viên xuất sắc của tháng" dựa trên đánh giá của khách hàng nữa.
[08:30] Dạ, em định là mình sẽ để một cái mã QR ở mỗi bàn á anh, khách họ quét mã đó để đánh giá phục vụ luôn, nếu bạn nào được khen nhiều thì mình thưởng nóng luôn cho tụi nhỏ nó có động lực.
[08:32] Hay đó, vầy mới đúng là cái anh cần nè. Công nghệ vô chút cho nó hiện đại. Team IT bên mình có làm cái này được không ta?
[08:34] Dạ được anh, cái này đơn giản mà, để em báo bên đó làm cái form khảo sát rồi tạo mã QR cho từng chi nhánh luôn.
[08:36] Ok, vậy chốt lại mấy cái đó nha. Mà Lan nè, em nhớ nhắc mấy bạn nhân viên là tuyệt đối không được dùng điện thoại trong giờ làm việc trừ trường hợp khẩn cấp nha, anh thấy cái này là cái gây khó chịu nhất cho khách luôn á.
[08:38] Dạ em biết rồi anh, em sẽ ra cái quy định mới và áp dụng hình thức kỷ luật nghiêm nếu bạn nào vi phạm cái này.
"""

# Đưa text trống vào biến số 2 (do chúng ta đang bắt model dự đoán đáp án)
prompt = format_prompt(input_text=test_meeting_text, output_text="")

# Hiển thị chuỗi prompt nguyên thủy sẽ đi vào não AI
print(prompt)

Hãy cập nhật lại báo cáo cuộc họp trước đó bằng cách bổ sung thêm thông tin từ nội dung thảo luận mới dưới đây.

### Đầu vào (Báo cáo cũ & Nội dung thảo luận mới):
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Tốc độ lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| Soạn lại giáo trình đào tạo kỹ năng mềm cho nhân viên | Chị Lan (Phòng Nhân sự) | 25

### Bước 4: Chạy Inference (Sinh đáp án)

In [11]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,        # Số tự tối đa cho bản tóm tắt
        temperature=0.3,           # Sinh chữ ổn định (ít sáng tạo bay bổng)
        top_p=0.9,                 # Lọc bớt các từ lạ
        repetition_penalty=1.1,    # Chống lặp từ
        pad_token_id=tokenizer.pad_token_id,
    )

# Cắt bỏ phần câu hỏi Prompt ở đầu, chỉ lấy phần chữ sinh ra ở đuôi
generated_tokens = outputs[0, input_length:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("====== BẢN TÓM TẮT AI TẠO RA ======")
print(result)

====== BẢN TÓM TẮT AI TẠO RA ======
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Giải quyết các khiếu nại của khách hàng về thái độ phục vụ và tốc độ lên món.
- Thống nhất các biện pháp cải thiện quy trình vận hành và nâng cao tính chuyên nghiệp của đội ngũ nhân viên.

### 2. Các vấn đề đã thảo luận
- Khách hàng phản hồi tiêu cực về thái độ của nhân viên (sử dụng điện thoại, thiếu chào hỏi) tại quận 1 và quận 3.
- Tốc độ lên món chậm (đợi 15-20 phút), nguyên nhân do quy trình quầy bar chưa tối ưu và nhân viên mới thiếu kỹ năng.
- Sự bất cập trong việc vệ sinh khu vực nhà vệ sinh và không gian quán tổng thể.
- Đề xuất sử dụng công nghệ qua mã QR để lấy ý kiến trực tiếp từ khách hàng.

### 3. Kết luận và quyết định
- Xây dựng lại bộ quy chuẩn đào tạo nhân sự, bổ sung kỹ năng mềm và tình huống thực tế.
- Thiết kế lại quầy bar theo mô hình dây chuyền một chiều để tối ưu hóa thao tác.
- Triển khai hệ thống điểm danh qua mã QR để đánh